In [ ]:




### **Code Cell 1: Environment Setup & Data Loading**

*(Copy this into a Colab Code cell)*

```python
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Mount Google Drive to access the dataset link provided in the project document
from google.colab import drive
drive.mount('/content/drive')

# NOTE: After mounting, update the file path below to match where you saved the dataset
# from the provided link: https://drive.google.com/drive/folders/10BRgPip2Zj_56is3DilJCowjfyT6E9AM
file_path = '/content/drive/MyDrive/crypto_data.csv' # UPDATE THIS PATH

try:
    df = pd.read_csv(file_path)
    print("Dataset loaded successfully!")
    display(df.head())
except FileNotFoundError:
    print("Please update the 'file_path' variable to point to the downloaded CSV file in your Google Drive.")

    # Creating a dummy dataset to demonstrate the pipeline in case the file isn't linked yet
    dates = pd.date_range(start='2016-01-01', end='2017-12-31')
    df = pd.DataFrame({
        'Date': dates,
        'Open': np.random.uniform(100, 1000, len(dates)),
        'High': np.random.uniform(105, 1050, len(dates)),
        'Low': np.random.uniform(90, 990, len(dates)),
        'Close': np.random.uniform(100, 1000, len(dates)),
        'Volume': np.random.uniform(10000, 500000, len(dates))
    })
    print("Generated sample data for pipeline demonstration.")

```

---

### **Code Cell 2: Data Preprocessing & Feature Engineering**

*(Copy this into a Colab Code cell)*

```python
# 1. Handle missing values and ensure data consistency
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date')
df.fillna(method='ffill', inplace=True) # Forward fill for time-series consistency

# 2. Engineer new features related to market liquidity trends
# Volatility: Difference between High and Low price
df['Volatility'] = df['High'] - df['Low']
# Avoid division by zero
df['Volatility'] = df['Volatility'].replace(0, 0.001)

# Moving Averages for trend analysis
df['MA_7'] = df['Close'].rolling(window=7).mean()
df['MA_30'] = df['Close'].rolling(window=30).mean()

# Liquidity Ratio (Target Variable Proxy): Higher volume with lower volatility = higher liquidity
df['Liquidity_Proxy'] = df['Volume'] / df['Volatility']

# Drop NaN values generated by moving averages
df.dropna(inplace=True)

print("Feature Engineering Complete. Dataset Shape:", df.shape)
df.head()

```

---

### **Code Cell 3: Exploratory Data Analysis (EDA)**

*(Copy this into a Colab Code cell)*

```python
plt.figure(figsize=(15, 10))

# 1. Price Trend
plt.subplot(2, 2, 1)
plt.plot(df['Date'], df['Close'], color='blue')
plt.title('Cryptocurrency Closing Price (2016-2017)')
plt.xlabel('Date')
plt.ylabel('Price')
plt.xticks(rotation=45)

# 2. Liquidity Trend
plt.subplot(2, 2, 2)
plt.plot(df['Date'], df['Liquidity_Proxy'], color='green')
plt.title('Estimated Market Liquidity Trend')
plt.xlabel('Date')
plt.ylabel('Liquidity Ratio')
plt.xticks(rotation=45)

# 3. Correlation Heatmap
plt.subplot(2, 2, 3)
corr_matrix = df[['Open', 'High', 'Low', 'Close', 'Volume', 'Volatility', 'Liquidity_Proxy']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlation Matrix')

plt.tight_layout()
plt.show()

```

---

### **Code Cell 4: Model Selection, Training & Scaling**

*(Copy this into a Colab Code cell)*

```python
# Select numerical features for modeling
features = ['Open', 'High', 'Low', 'Close', 'Volume', 'Volatility', 'MA_7', 'MA_30']
X = df[features]
y = df['Liquidity_Proxy'] # Predicting liquidity

# Train-Test Split (80% Training, 20% Testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=False)

# Normalize and scale numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Initialize and train the Machine Learning Model
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

print("Model Training Completed Successfully!")

```

---

### **Code Cell 5: Model Evaluation & Hyperparameter Tuning**

*(Copy this into a Colab Code cell)*

```python
# Make Predictions on unseen data
y_pred = model.predict(X_test_scaled)

# Assess model performance using metrics
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Initial Model Evaluation ---")
print(f"RMSE (Root Mean Squared Error): {rmse:.4f}")
print(f"MAE (Mean Absolute Error): {mae:.4f}")
print(f"R² Score: {r2:.4f}")

# Hyperparameter Tuning (Optimizing model parameters for better accuracy)
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

print("\nStarting Hyperparameter Tuning (this may take a minute)...")
grid_search = GridSearchCV(RandomForestRegressor(random_state=42), param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

best_model = grid_search.best_estimator_
best_y_pred = best_model.predict(X_test_scaled)

best_rmse = np.sqrt(mean_squared_error(y_test, best_y_pred))
best_mae = mean_absolute_error(y_test, best_y_pred)
best_r2 = r2_score(y_test, best_y_pred)

print("\n--- Tuned Model Evaluation ---")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Optimized RMSE: {best_rmse:.4f}")
print(f"Optimized MAE: {best_mae:.4f}")
print(f"Optimized R² Score: {best_r2:.4f}")


### **Code Cell 6: Visualizing Predictions vs Actual Liquidity**

*(Copy this into a Colab Code cell)*

```python
plt.figure(figsize=(12, 6))
plt.plot(df['Date'].iloc[-len(y_test):], y_test.values, label='Actual Liquidity', color='blue', alpha=0.6)
plt.plot(df['Date'].iloc[-len(y_test):], best_y_pred, label='Predicted Liquidity', color='red', alpha=0.8, linestyle='--')
plt.title('Actual vs Predicted Cryptocurrency Liquidity')
plt.xlabel('Date')
plt.ylabel('Liquidity Score')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

```

### Local Deployment Note

To complete the "Local Deployment" requirement using Flask or Streamlit, you will need to save the scaler and model locally in Colab, download them, and run the Streamlit app on your local machine:

```python
# Run this in Colab to export your model
import joblib
joblib.dump(best_model, 'liquidity_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("Model and scaler saved. Download these files to use in your local Streamlit app.")

```